# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset with mlcroissant
dataset = mlc.Dataset(url)

# Show dataset metadata
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'unknown')}")
print(f"License: {getattr(metadata, 'license', 'unknown')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets with their @id and name
print("Available record sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"@id: {rs.id} - name: {getattr(rs, 'name', 'N/A')}")

# For this dataset, the main data is typically in the primary record set.
# We'll print fields for each record set.
for rs in record_sets:
    print(f"\nFields for record set '@id: {rs.id}':")
    for field in rs.fields:
        print(f"  - @id: {field.id:50s} | Name: {field.name:20s} | dataType: {getattr(field, 'dataType', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Select record set @id of interest (edit as needed referencing the printed @id above).
# For this dataset, record sets often contain identifiers such as:
#   'https://api.app.sen.science/frontiers/7154140/records/protein_data'
# Adjust the @id value below to the actual main @id printed previously.
main_record_set_id = dataset.record_sets[0].id

# If the dataset has further record_sets, add their @id here as needed.
record_sets_ids = [main_record_set_id]
dataframes = {}
for rs_id in record_sets_ids:
    # Load into DataFrame
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

print(f"Loaded columns from record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
df = dataframes[main_record_set_id]

# Print column info for numerical field selection
print("Columns in DataFrame:")
print(df.columns.tolist())

# Choose a numeric field for analysis — e.g., 'abundance' or 'MW' (molecular weight) if available
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) or df[col].dtype in ['float64', 'int64']]
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Selected numeric field: {numeric_field}")
else:
    # Fallback — try to use a likely field name
    for possible in ['MW', 'abundance', 'Coverage', 'Peptide_count']:
        if possible in df.columns:
            numeric_field = possible
            break
    else:
        raise RuntimeError("No numeric field found for analysis. Please check column names.")

# Filtering — example: retain records with numeric_field > threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalization (Z transformation)
normalized_col = f"{numeric_field}_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, normalized_col]].head())

# Group by categorical field, e.g. 'Sample', 'Protein_Class', or another
group_candidates = [col for col in df.columns if col != numeric_field and df[col].dtype == 'object']
group_field = group_candidates[0] if group_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f"Mean_{numeric_field}")
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.figsize'] = (8, 5)

# Histogram of selected numeric field
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If group_field exists, boxplot
if group_field:
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides mass spectrometry-based protein abundance and metadata for extracellular vesicles from human mast cells.
- We demonstrated loading Croissant metadata and tabular data using `mlcroissant`, exploring columns and their types via `@id` references.
- We performed basic filtering, normalization, grouping, and visualizations on a numeric field of interest.

For more in-depth analysis, use additional field `@id`s and integrate downstream machine learning as appropriate.